# SGA-2025 Galaxy Embeddings: Exploring the Embedding Space

The [SGA-2025](https://github.com/moustakas/SGA) catalog contains ~350,000 large nearby
galaxies imaged in grz by the DESI Legacy Survey.  This notebook lets you explore a
2048-dimensional **self-supervised embedding space** computed from those images using a
pre-trained MoCo v2 ResNet50 model
([Stein et al. 2022](https://arxiv.org/abs/2110.13151)).

The embeddings encode visual morphology — galaxies that look similar tend to cluster
together even though the model was never told anything about galaxy types.

**What this notebook covers:**
1. Load the pre-computed embeddings and the SGA-2025 catalog
2. Explore catalog properties of the sample
3. Reduce the 2048-d embedding space to 2-d with PCA and UMAP
4. Examine the actual galaxies in different regions of the embedding space
5. Find morphological nearest neighbors of individual galaxies

---
> Cells marked **▶ Try this** are starting points for open-ended exploration.
> The catalog cut in Section 2 is a natural place to diverge and see how different
> sample selections change the structure of the embedding space.

## 0. Setup

In [ ]:
import os
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap

from SGA.SGA import read_sga_sample
from SGA.ssl import load_ssl_embeddings
from SGA.qa import notebook_style, sdss_rgb

In [ ]:
# Paths — adjust if running outside NERSC
REGION  = 'dr11-south'
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'

notebook_style()

## 1. Load the data

`load_ssl_embeddings` reads the pre-computed HDF5 file and joins it to the SGA-2025
catalog on SGAID, returning a single table with all catalog columns plus
`embeddings` (2048-d) and `projections` (128-d) for each galaxy.

In [ ]:
data = load_ssl_embeddings(REGION, SSL_DIR)
print(f'Loaded {len(data):,d} galaxies')
print(f'Embedding dimension:   {data["embeddings"].shape[1]}')
print(f'Projection dimension:  {data["projections"].shape[1]}')
print(f'Catalog columns: {data.colnames}')

In [ ]:
# Quick look at the table
data[['SGAID', 'GROUP_NAME', 'RA', 'DEC', 'D26', 'BA', 'BANDS']][:8]

### Image display helpers

These functions let you view actual galaxy cutouts from the HDF5 dataset.
Building the cutout index scans all chunk files to map SGAID → file location;
it takes about 30 seconds on the full sample.
Cutout titles show (RA, Dec) by default for easy cross-referencing with the
[Legacy Survey viewer](https://www.legacysurvey.org/viewer).

In [ ]:
def build_cutout_index(ssl_dir, region):
    """Return {sgaid: (hdf5_path, row_index)} for fast image retrieval."""
    files = sorted(glob.glob(os.path.join(ssl_dir, f'ssl-cutouts-{region}-chunk*.hdf5')))
    if not files:
        raise FileNotFoundError(f'No cutout files found in {ssl_dir}')
    index = {}
    for f in files:
        with h5py.File(f, 'r') as H:
            for i, sgaid in enumerate(H['sgaid'][:]):
                index[int(sgaid)] = (f, i)
    return index


def show_cutout_grid(sgaids, cutout_index, ncols=5, figsize_per=2.5,
                     titles=None, data=None):
    """Display a grid of galaxy cutouts given a list of SGAIDs.

    If `data` is provided and `titles` is None, each panel is labelled
    with (RA, Dec) from the catalog for easy viewer cross-referencing.
    """
    sgaids = [int(s) for s in sgaids]

    # Build RA/Dec lookup from the catalog table if supplied
    radec = {}
    if data is not None and titles is None:
        for s, ra, dec in zip(np.asarray(data['SGAID']),
                              np.asarray(data['RA']),
                              np.asarray(data['DEC'])):
            radec[int(s)] = (float(ra), float(dec))

    nrows = int(np.ceil(len(sgaids) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * figsize_per, nrows * figsize_per))
    axes = np.array(axes).ravel()

    for i, (ax, sgaid) in enumerate(zip(axes, sgaids)):
        if sgaid in cutout_index:
            fname, idx = cutout_index[sgaid]
            with h5py.File(fname, 'r') as H:
                img = H['images'][idx]          # (3, 152, 152)
            rgb = sdss_rgb([img[0], img[1], img[2]], ['g', 'r', 'z'])
            ax.imshow(rgb, origin='lower')
        else:
            ax.text(0.5, 0.5, f'SGAID\n{sgaid}\nnot found',
                    ha='center', va='center', transform=ax.transAxes, fontsize=7)

        if titles is not None:
            title = str(titles[i])
        elif sgaid in radec:
            ra, dec = radec[sgaid]
            title = f'({ra:.4f}, {dec:.4f})'
        else:
            title = str(sgaid)
        ax.set_title(title, fontsize=7)
        ax.axis('off')

    for ax in axes[len(sgaids):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


print('Building cutout index...')
cutout_index = build_cutout_index(SSL_DIR, REGION)
print(f'Done: {len(cutout_index):,d} objects indexed')

In [ ]:
# Display 25 random galaxies to get oriented
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(data), size=25, replace=False)
show_cutout_grid(data['SGAID'][sample_idx], cutout_index, ncols=5, data=data)

## 2. Explore the sample

Before doing any dimensionality reduction, it helps to understand what kinds of
galaxies are in the dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Diameter distribution
axes[0].hist(np.log10(np.asarray(data['D26'])), bins=60,
             color='steelblue', edgecolor='none')
axes[0].set_xlabel(r'$\log_{10}$(D$_{26}$ [arcmin])')
axes[0].set_ylabel('Number of galaxies')
axes[0].set_title('Size distribution')

# Axis ratio (b/a)
ba = np.asarray(data['BA'])
axes[1].hist(ba[(ba > 0) & (ba <= 1)], bins=50, color='coral', edgecolor='none')
axes[1].set_xlabel('Axis ratio  b/a')
axes[1].set_title('Shape distribution')

# Band coverage
from collections import Counter
band_counts = Counter(np.asarray(data['BANDS']))
labels, counts = zip(*sorted(band_counts.items(), key=lambda x: -x[1]))
axes[2].bar(labels, counts, color='mediumseagreen', edgecolor='none')
axes[2].set_xlabel('Available bands')
axes[2].set_title('Imaging band coverage')

plt.tight_layout()
plt.show()

### Make a catalog cut

The cell below applies an example cut.  Modify it to select whatever subset interests you.

Some ideas:
- **By size**: `data['D26'] > 1.0` — only galaxies larger than 1 arcmin
- **By shape**: `data['BA'] < 0.4` — edge-on; `data['BA'] > 0.85` — round/face-on
- **By imaging**: `np.char.startswith(np.asarray(data['BANDS']), 'grz')` — grz only
- **By group membership**: `np.asarray(data['GROUP_MULT']) == 1` — isolated galaxies only
- **No cut at all**: use the full sample and see what structure emerges

In [ ]:
# ── Modify this cut ──────────────────────────────────────────────────────────
mask = np.ones(len(data), dtype=bool)   # start with the full sample
# mask = data['D26'] > 0.5              # example: D26 > 0.5 arcmin
# ─────────────────────────────────────────────────────────────────────────────

subset = data[mask]
print(f'Keeping {len(subset):,d} / {len(data):,d} galaxies ({100*len(subset)/len(data):.1f}%)')

## 3. Dimensionality reduction

The embeddings live in 2048 dimensions — impossible to visualise directly.
We project down to 2-d using two complementary methods:

| Method | Speed | What it preserves |
|--------|-------|-------------------|
| PCA    | Fast, deterministic | Global variance structure |
| UMAP   | Slower, stochastic  | Local neighborhoods and clusters |

We work with a random subsample to keep computation fast.
Adjust `N_SUBSAMPLE` or replace random sampling with a catalog-driven selection.

In [ ]:
N_SUBSAMPLE = 50_000   # ← adjust as you like

rng = np.random.default_rng(42)
if len(subset) > N_SUBSAMPLE:
    idx = rng.choice(len(subset), size=N_SUBSAMPLE, replace=False)
    sub = subset[idx]
else:
    sub = subset

emb = np.array(sub['embeddings'])   # (N, 2048)
print(f'Working with {len(sub):,d} galaxies, embedding matrix shape: {emb.shape}')

### PCA

In [ ]:
print('Running PCA...')
pca = PCA(n_components=10, random_state=42)
pca_coords = pca.fit_transform(emb)   # (N, 10)

var = pca.explained_variance_ratio_
print(f'Variance explained — PC1: {100*var[0]:.1f}%  '
      f'PC2: {100*var[1]:.1f}%  '
      f'top-10 total: {100*var.sum():.1f}%')

In [ ]:
# ── Change color_by to explore different catalog properties ───────────────────
color_by    = np.log10(np.asarray(sub['D26']))
color_label = r'$\log_{10}$(D$_{26}$ [arcmin])'
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(pca_coords[:, 0], pca_coords[:, 1],
                c=color_by, s=1, alpha=0.4, cmap='plasma', rasterized=True)
plt.colorbar(sc, ax=ax, label=color_label)
ax.set_xlabel(f'PC 1  ({100*var[0]:.1f}% var.)')
ax.set_ylabel(f'PC 2  ({100*var[1]:.1f}% var.)')
ax.set_title(f'PCA — {len(sub):,d} galaxies')
plt.tight_layout()
plt.show()

### UMAP

UMAP often reveals cluster structure invisible in PCA.  Two key parameters control
the result — experiment with both:

- **`n_neighbors`** (default 15): how much local vs. global structure to preserve.  
  Small values (5–15) emphasise tight local clusters; large values (50–200) show
  broader global organisation.
- **`min_dist`** (default 0.1): how tightly points are packed.  
  Values near 0 produce dense clumps; values near 1 spread them out.

**Background reading:**
- [McInnes et al. 2018 — original UMAP paper](https://arxiv.org/abs/1802.03426)
- [Understanding UMAP — interactive visual explainer](https://pair-code.github.io/understanding-umap/)
- [UMAP parameters documentation](https://umap-learn.readthedocs.io/en/latest/parameters.html)

In [ ]:
# ── Experiment with these parameters ─────────────────────────────────────────
N_NEIGHBORS = 15       # try 5, 30, 100, 200
MIN_DIST    = 0.1      # try 0.0, 0.1, 0.5, 0.9
METRIC      = 'cosine' # try 'euclidean'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Running UMAP (n_neighbors={N_NEIGHBORS}, min_dist={MIN_DIST}, metric={METRIC!r})...')
reducer = umap.UMAP(n_components=2, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST,
                    metric=METRIC, random_state=42, low_memory=True)
umap_coords = reducer.fit_transform(emb)   # (N, 2)
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(umap_coords[:, 0], umap_coords[:, 1],
                c=color_by, s=1, alpha=0.4, cmap='plasma', rasterized=True)
plt.colorbar(sc, ax=ax, label=color_label)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title(f'UMAP — {len(sub):,d} galaxies  '
             f'(n_neighbors={N_NEIGHBORS}, min_dist={MIN_DIST})')
plt.tight_layout()
plt.show()

**▶ Try this:**
1. Change `color_by` to a different column (`BA`, `PA`, `BANDS`, or something else).  
   What structure does the embedding space seem to encode?
2. Re-run UMAP with `emb = np.array(sub['projections'])` (128-d) instead of the 2048-d backbone.  
   Is the structure different? Why might that be?
3. Try two or three different values of `N_NEIGHBORS`. What changes? What stays the same?

## 4. Inspect galaxies in the embedding space

Pick a region of the UMAP that looks interesting and display the actual galaxies there.
Adjust the coordinate bounds below to select whatever region you want to examine.

In [ ]:
# ── Change these bounds to select a different region of the UMAP ─────────────
x1, x2 = umap_coords[:, 0].min(), umap_coords[:, 0].min() + 2.0
y1, y2 = umap_coords[:, 1].max() - 2.0, umap_coords[:, 1].max()
# ─────────────────────────────────────────────────────────────────────────────

in_region = ((umap_coords[:, 0] > x1) & (umap_coords[:, 0] < x2) &
             (umap_coords[:, 1] > y1) & (umap_coords[:, 1] < y2))
region_sgaids = np.asarray(sub['SGAID'])[in_region]
print(f'{in_region.sum()} galaxies in selected region')

# Overplot selection on UMAP
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(umap_coords[~in_region, 0], umap_coords[~in_region, 1],
           c='lightgray', s=1, alpha=0.3, rasterized=True)
ax.scatter(umap_coords[in_region, 0], umap_coords[in_region, 1],
           c='crimson', s=4, alpha=0.8, label='selected', rasterized=True)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Display up to 25 galaxies from the selected region
show_cutout_grid(region_sgaids[:25], cutout_index, ncols=5, data=sub)

**▶ Try this:**  
Find a region of the UMAP that looks interesting — a tight cluster, an isolated
group, or somewhere with an unusual color.  Modify the coordinate bounds and
look at the galaxies there.  What do they have in common?
Are there any that look out of place?

## 5. Similarity search

Given any galaxy, we can find its nearest neighbors in embedding space using
cosine similarity.  Galaxies that are close in this space should look visually similar,
even if they are far apart on the sky.

In [ ]:
def find_neighbors(query_sgaid, table, n=20, use='projections'):
    """Return the SGAIDs and cosine similarities of the N nearest neighbors.

    Parameters
    ----------
    query_sgaid : int
    table : astropy.table.Table with 'SGAID' and `use` columns
    n : int
    use : 'projections' (128-d, contrastive space) or 'embeddings' (2048-d backbone)
    """
    sgaids = np.asarray(table['SGAID'])
    match  = sgaids == int(query_sgaid)
    if not np.any(match):
        raise ValueError(f'SGAID {query_sgaid} not found in table')

    vecs  = np.array(table[use], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-10
    vecs  = vecs / norms

    query = vecs[match][0]
    sims  = vecs @ query

    order = np.argsort(sims)[::-1]
    order = order[sgaids[order] != int(query_sgaid)][:n]
    return sgaids[order], sims[order]

In [ ]:
# ── Pick any galaxy by SGAID ──────────────────────────────────────────────────
# To find a specific galaxy by name:
#   np.asarray(data['SGAID'])[np.asarray(data['GROUP_NAME']) == 'NGC1068']
query_sgaid = int(data['SGAID'][0])    # ← change this
N_NEIGHBORS = 20
# ─────────────────────────────────────────────────────────────────────────────

neighbor_sgaids, sims = find_neighbors(query_sgaid, data, n=N_NEIGHBORS, use='projections')

query_name = str(np.asarray(data['GROUP_NAME'])[np.asarray(data['SGAID']) == query_sgaid][0])
print(f'Query: {query_name} (SGAID {query_sgaid})')
print(f'Top-5 cosine similarities: {sims[:5].round(4)}')

print('\nQuery galaxy:')
show_cutout_grid([query_sgaid], cutout_index, ncols=1, figsize_per=4, data=data)

print(f'\n{N_NEIGHBORS} nearest neighbors:')
show_cutout_grid(neighbor_sgaids, cutout_index, ncols=5, data=data)

**▶ Try this:**
1. Pick a galaxy you find visually interesting — something round, something edge-on,
   a group member — and find its nearest neighbors.  Do they look similar?
2. Change `use='embeddings'` (2048-d) instead of `'projections'` (128-d).  
   Do you get different neighbors?  Which gives more visually coherent results?
3. Pick two galaxies that look very different from each other and compare their
   neighbor lists.  What does this tell you about what the model has learned to
   distinguish?
4. **(Advanced)** For a larger search, `faiss-cpu` is available in the SGAML environment.
   Build a Faiss flat index over the full `data['projections']` array for
   sub-second nearest-neighbor queries across all ~260,000 galaxies.